# Ask-SLM Runner
Minimal notebook flow: aggregate view inputs, prepare mappings, build analysis-wide dataset, and launch managers.

In [1]:
from pathlib import Path
import os
import sys

import pandas as pd

if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

from managers.runner_pipeline_manager import (
    build_or_load_tax_accessor,
    enrich_envo_columns,
    initialize_runner_context,
    prepare_workspace_inputs,
    refresh_mapping_templates,
    run_enrichment_stage,
    run_merge_stage,
    run_taxonomy_stage,
)


def require_vars(names):
    missing = [name for name in names if name not in globals()]
    if missing:
        raise RuntimeError(f"Missing required variables: {', '.join(missing)}")

In [2]:
view_name = "IBDProjects"

context, runner_logger = initialize_runner_context(view_name=view_name)

# Convenience aliases for downstream cells.
results_dir = context.results_dir
config_dir = context.config_dir


def log_run(message: str, echo: bool = False):
    runner_logger.log(message, echo=echo)

[2026-04-16 11:01:07] Log file path: Workspaces/IBDProjects/Logs/run_20260416_110107_653780.log


In [3]:
metadata_table, taxonomy_table, metadata_source, taxonomy_source = prepare_workspace_inputs(
    context=context,
    logger=runner_logger,
)

display(metadata_table.head())

,acc,assay_type,center_name,consent,experiment,sample_name,instrument,librarylayout,libraryselection,librarysource,...,ena_first_public_run,ena_last_update_run,sample_name_sam,datastore_filetype,datastore_provider,datastore_region,attributes,jattr,run_file_version,bioproject
0,SRR27827344,WGS,"VEDANTA BIOSCIENCES, INC.",public,SRX23490556,CY11557.fastq.gz,Illumina NovaSeq 6000,SINGLE,RANDOM,METAGENOMIC,...,NaN,NaN,NaN,"[run.zq, sra]","[gs, ncbi, s3]","[gs.us-east1, ncbi.public, s3.us-east-1]","[{k=bases, v=284876257}, {k=bytes, v=77936898}...","{""bases"": 284876257, ""bytes"": 77936898, ""run_f...",2.0,PRJNA1071720
1,SRR27827258,WGS,"VEDANTA BIOSCIENCES, INC.",public,SRX23490642,CY17560.fastq.gz,Illumina NovaSeq 6000,SINGLE,RANDOM,METAGENOMIC,...,NaN,NaN,NaN,"[run.zq, sra]","[gs, ncbi, s3]","[gs.us-east1, ncbi.public, s3.us-east-1]","[{k=bases, v=266418103}, {k=bytes, v=72665111}...","{""bases"": 266418103, ""bytes"": 72665111, ""run_f...",2.0,PRJNA1071720
2,SRR27827360,WGS,"VEDANTA BIOSCIENCES, INC.",public,SRX23490540,CY11530.fastq.gz,Illumina NovaSeq 6000,SINGLE,RANDOM,METAGENOMIC,...,NaN,NaN,NaN,"[run.zq, sra]","[gs, ncbi, s3]","[gs.us-east1, ncbi.public, s3.us-east-1]","[{k=bases, v=172496991}, {k=bytes, v=44458429}...","{""bases"": 172496991, ""bytes"": 44458429, ""run_f...",2.0,PRJNA1071720
3,SRR27827171,WGS,"VEDANTA BIOSCIENCES, INC.",public,SRX23490729,CY17384.fastq.gz,Illumina NovaSeq 6000,SINGLE,RANDOM,METAGENOMIC,...,NaN,NaN,NaN,"[run.zq, sra]","[gs, ncbi, s3]","[gs.us-east1, ncbi.public, s3.us-east-1]","[{k=bases, v=654865820}, {k=bytes, v=123505477...","{""bases"": 654865820, ""bytes"": 123505477, ""run_...",2.0,PRJNA1071720
4,SRR27827339,WGS,"VEDANTA BIOSCIENCES, INC.",public,SRX23490561,CY19656.fastq.gz,Illumina NovaSeq 6000,SINGLE,RANDOM,METAGENOMIC,...,NaN,NaN,NaN,"[run.zq, sra]","[gs, ncbi, s3]","[gs.us-east1, ncbi.public, s3.us-east-1]","[{k=bases, v=271688182}, {k=bytes, v=76565240}...","{""bases"": 271688182, ""bytes"": 76565240, ""run_f...",2.0,PRJNA1071720


In [4]:
sra_processor, project_ids, project_status_df = refresh_mapping_templates(
    metadata_df=metadata_table,
    context=context,
    logger=runner_logger,
)

display(project_status_df)

,project_id,config_exists,config_path
0,PRJNA1071720,True,Workspaces/IBDProjects/mapping_configs/PRJNA10...
1,PRJNA389280,True,Workspaces/IBDProjects/mapping_configs/PRJNA38...
2,PRJNA398089,True,Workspaces/IBDProjects/mapping_configs/PRJNA39...


In [5]:
from managers.MappingConfigManager.mapping_config_manager import MappingConfigManager

mapping_manager = MappingConfigManager(
    config_dir=context.config_dir,
    logger=runner_logger.log,
)
mapping_manager.run()

✅ Loaded 27 fields from Metalog schema
📋 Loaded 3 project(s)
📝 Found 1 existing target field(s): ['environment_feature']



In [6]:
enriched_metadata, mapped_cols, missing_mapping_projects, enrichment_current_stamp = run_enrichment_stage(
    metadata_df=metadata_table,
    processor=sra_processor,
    project_ids=project_ids,
    context=context,
    logger=runner_logger,
    strict_missing_configs=False,
)

enriched_metadata, envo_report_df = enrich_envo_columns(
    metadata_df=enriched_metadata,
    context=context,
    logger=runner_logger,
)

summary_df = pd.DataFrame(
    {
        "metric": ["samples", "projects", "mapped_columns", "projects_missing_mapping_config", "envo_terms_resolved"],
        "value": [
            len(enriched_metadata),
            enriched_metadata["bioproject"].nunique(),
            len(mapped_cols),
            len(missing_mapping_projects),
            len(envo_report_df),
        ],
    }
)

display(summary_df)

if missing_mapping_projects:
    display(
        pd.DataFrame(
            {
                "missing_project_id": missing_mapping_projects,
                "expected_config_path": [str(context.config_dir / f"{p}.json") for p in missing_mapping_projects],
                "template_path": [str(context.config_dir / f"{p}.json.template") for p in missing_mapping_projects],
            }
        )
    )

if not envo_report_df.empty:
    display(envo_report_df.head(20))

display(enriched_metadata.head())

[2026-04-16 11:01:11] Enrichment cache hit: loaded from Workspaces/IBDProjects/results/_cache


,metric,value
0,samples,4732
1,projects,3
2,mapped_columns,1
3,projects_missing_mapping_config,0
4,envo_terms_resolved,0


,acc,assay_type,center_name,consent,experiment,sample_name,instrument,librarylayout,libraryselection,librarysource,...,ena_last_update_run,sample_name_sam,datastore_filetype,datastore_provider,datastore_region,attributes,jattr,run_file_version,bioproject,environment_feature
0,SRR27827344,WGS,"VEDANTA BIOSCIENCES, INC.",public,SRX23490556,CY11557.fastq.gz,Illumina NovaSeq 6000,SINGLE,RANDOM,METAGENOMIC,...,NaN,NaN,"[run.zq, sra]","[gs, ncbi, s3]","[gs.us-east1, ncbi.public, s3.us-east-1]","[{k=bases, v=284876257}, {k=bytes, v=77936898}...","{""bases"": 284876257, ""bytes"": 77936898, ""run_f...",2.0,PRJNA1071720,NaN
1,SRR27827258,WGS,"VEDANTA BIOSCIENCES, INC.",public,SRX23490642,CY17560.fastq.gz,Illumina NovaSeq 6000,SINGLE,RANDOM,METAGENOMIC,...,NaN,NaN,"[run.zq, sra]","[gs, ncbi, s3]","[gs.us-east1, ncbi.public, s3.us-east-1]","[{k=bases, v=266418103}, {k=bytes, v=72665111}...","{""bases"": 266418103, ""bytes"": 72665111, ""run_f...",2.0,PRJNA1071720,NaN
2,SRR27827360,WGS,"VEDANTA BIOSCIENCES, INC.",public,SRX23490540,CY11530.fastq.gz,Illumina NovaSeq 6000,SINGLE,RANDOM,METAGENOMIC,...,NaN,NaN,"[run.zq, sra]","[gs, ncbi, s3]","[gs.us-east1, ncbi.public, s3.us-east-1]","[{k=bases, v=172496991}, {k=bytes, v=44458429}...","{""bases"": 172496991, ""bytes"": 44458429, ""run_f...",2.0,PRJNA1071720,NaN
3,SRR27827171,WGS,"VEDANTA BIOSCIENCES, INC.",public,SRX23490729,CY17384.fastq.gz,Illumina NovaSeq 6000,SINGLE,RANDOM,METAGENOMIC,...,NaN,NaN,"[run.zq, sra]","[gs, ncbi, s3]","[gs.us-east1, ncbi.public, s3.us-east-1]","[{k=bases, v=654865820}, {k=bytes, v=123505477...","{""bases"": 654865820, ""bytes"": 123505477, ""run_...",2.0,PRJNA1071720,NaN
4,SRR27827339,WGS,"VEDANTA BIOSCIENCES, INC.",public,SRX23490561,CY19656.fastq.gz,Illumina NovaSeq 6000,SINGLE,RANDOM,METAGENOMIC,...,NaN,NaN,"[run.zq, sra]","[gs, ncbi, s3]","[gs.us-east1, ncbi.public, s3.us-east-1]","[{k=bases, v=271688182}, {k=bytes, v=76565240}...","{""bases"": 271688182, ""bytes"": 76565240, ""run_f...",2.0,PRJNA1071720,NaN


In [7]:
TAXON_KEY_SEP = "||"

abundance_matrix, taxon_lookup_df, current_taxonomy_stamp = run_taxonomy_stage(
    taxonomy_df=taxonomy_table,
    context=context,
    logger=runner_logger,
    taxon_key_sep=TAXON_KEY_SEP,
)

abundance_metadata_matrix, metadata_columns, current_merge_stamp = run_merge_stage(
    enriched_metadata_df=enriched_metadata,
    abundance_matrix=abundance_matrix,
    enrichment_stamp=enrichment_current_stamp,
    taxonomy_stamp=current_taxonomy_stamp,
    context=context,
    logger=runner_logger,
)

artifacts_df = pd.DataFrame(
    {
        "artifact": [
            "abundance_matrix",
            "abundance_metadata_matrix",
            "taxon_lookup",
            "metadata_columns",
            "taxonomy_stage_stamp",
            "merge_stage_stamp",
            "combined_stamp",
            "envo_dictionary",
        ],
        "path": [
            str(context.results_dir / "Ask_SLM_Taxon_Abundance_Matrix.csv"),
            str(context.results_dir / "Ask_SLM_Analysis_Wide_Dataset.csv"),
            str(context.results_dir / "Ask_SLM_Taxon_Lookup.csv"),
            str(context.results_dir / "Ask_SLM_Metadata_Columns.json"),
            str(context.cache_dir / "taxonomy_stage_stamp.json"),
            str(context.cache_dir / "merge_stage_stamp.json"),
            str(context.results_dir / "Ask_SLM_Cache_Stamp.json"),
            str(context.results_dir / "ENVO_Term_Dictionary.csv"),
        ],
    }
)

display(artifacts_df)
display(abundance_metadata_matrix[metadata_columns].head())

[2026-04-16 11:01:18] Taxonomy cache hit: loaded from Workspaces/IBDProjects/results
[2026-04-16 11:01:27] Merge cache hit: loaded from Workspaces/IBDProjects/results


/Users/yarintamam/Code/Ask-SLM/managers/runner_pipeline_manager.py:519: DtypeWarning: Columns (0: collection_date_sam, 1: environment_feature) have mixed types. Specify dtype option on import or set low_memory=False.
  abundance_metadata_matrix = pd.read_csv(canonical_dataset_path)


,artifact,path
0,abundance_matrix,Workspaces/IBDProjects/results/Ask_SLM_Taxon_A...
1,abundance_metadata_matrix,Workspaces/IBDProjects/results/Ask_SLM_Analysi...
2,taxon_lookup,Workspaces/IBDProjects/results/Ask_SLM_Taxon_L...
3,metadata_columns,Workspaces/IBDProjects/results/Ask_SLM_Metadat...
4,taxonomy_stage_stamp,Workspaces/IBDProjects/results/_cache/taxonomy...
5,merge_stage_stamp,Workspaces/IBDProjects/results/_cache/merge_st...
6,combined_stamp,Workspaces/IBDProjects/results/Ask_SLM_Cache_S...
7,envo_dictionary,Workspaces/IBDProjects/results/ENVO_Term_Dicti...


,acc,assay_type,center_name,consent,experiment,sample_name,instrument,librarylayout,libraryselection,librarysource,...,ena_last_update_run,sample_name_sam,datastore_filetype,datastore_provider,datastore_region,attributes,jattr,run_file_version,bioproject,environment_feature
0,SRR27827344,WGS,"VEDANTA BIOSCIENCES, INC.",public,SRX23490556,CY11557.fastq.gz,Illumina NovaSeq 6000,SINGLE,RANDOM,METAGENOMIC,...,NaN,NaN,"[run.zq, sra]","[gs, ncbi, s3]","[gs.us-east1, ncbi.public, s3.us-east-1]","[{k=bases, v=284876257}, {k=bytes, v=77936898}...","{""bases"": 284876257, ""bytes"": 77936898, ""run_f...",2.0,PRJNA1071720,NaN
1,SRR27827258,WGS,"VEDANTA BIOSCIENCES, INC.",public,SRX23490642,CY17560.fastq.gz,Illumina NovaSeq 6000,SINGLE,RANDOM,METAGENOMIC,...,NaN,NaN,"[run.zq, sra]","[gs, ncbi, s3]","[gs.us-east1, ncbi.public, s3.us-east-1]","[{k=bases, v=266418103}, {k=bytes, v=72665111}...","{""bases"": 266418103, ""bytes"": 72665111, ""run_f...",2.0,PRJNA1071720,NaN
2,SRR27827360,WGS,"VEDANTA BIOSCIENCES, INC.",public,SRX23490540,CY11530.fastq.gz,Illumina NovaSeq 6000,SINGLE,RANDOM,METAGENOMIC,...,NaN,NaN,"[run.zq, sra]","[gs, ncbi, s3]","[gs.us-east1, ncbi.public, s3.us-east-1]","[{k=bases, v=172496991}, {k=bytes, v=44458429}...","{""bases"": 172496991, ""bytes"": 44458429, ""run_f...",2.0,PRJNA1071720,NaN
3,SRR27827171,WGS,"VEDANTA BIOSCIENCES, INC.",public,SRX23490729,CY17384.fastq.gz,Illumina NovaSeq 6000,SINGLE,RANDOM,METAGENOMIC,...,NaN,NaN,"[run.zq, sra]","[gs, ncbi, s3]","[gs.us-east1, ncbi.public, s3.us-east-1]","[{k=bases, v=654865820}, {k=bytes, v=123505477...","{""bases"": 654865820, ""bytes"": 123505477, ""run_...",2.0,PRJNA1071720,NaN
4,SRR27827339,WGS,"VEDANTA BIOSCIENCES, INC.",public,SRX23490561,CY19656.fastq.gz,Illumina NovaSeq 6000,SINGLE,RANDOM,METAGENOMIC,...,NaN,NaN,"[run.zq, sra]","[gs, ncbi, s3]","[gs.us-east1, ncbi.public, s3.us-east-1]","[{k=bases, v=271688182}, {k=bytes, v=76565240}...","{""bases"": 271688182, ""bytes"": 76565240, ""run_f...",2.0,PRJNA1071720,NaN


In [8]:
from managers.AlphaDiversityManager.alpha_diversity_manager import AlphaDiversityManager

require_vars(["abundance_metadata_matrix", "taxon_lookup_df", "metadata_columns", "current_taxonomy_stamp"])

shared_tax_accessor = build_or_load_tax_accessor(
    taxon_lookup_df=taxon_lookup_df,
    taxonomy_stamp=current_taxonomy_stamp,
    context=context,
    logger=runner_logger,
)

alpha_manager = AlphaDiversityManager(
    analysis_wide_df=abundance_metadata_matrix,
    taxon_lookup_df=taxon_lookup_df,
    metadata_cols=metadata_columns,
    tax_accessor=shared_tax_accessor,
    logger=runner_logger.log,
)
alpha_manager.run()

[2026-04-16 11:01:27] Taxonomy accessor cache hit: loaded lineage DB from Workspaces/IBDProjects/results/_cache/taxonomy_lineage_db.json
✅ Loaded lineage database with 18022 taxa


In [9]:
from managers.TaxonomicAbundanceManager import TaxonomicAbundanceManager

require_vars(["abundance_metadata_matrix", "taxon_lookup_df", "metadata_columns"])

abundance_manager = TaxonomicAbundanceManager(
    abundance_metadata_matrix=abundance_metadata_matrix,
    taxon_lookup_df=taxon_lookup_df,
    metadata_cols=metadata_columns,
    logger=runner_logger.log,
)
abundance_manager.run()

In [10]:
# from managers.LongitudinalAnalysisManager import LongitudinalAnalysisManager

# require_vars(["enriched_df", "sra_metadata_df"])

# longitudinal_manager = LongitudinalAnalysisManager(enriched_metadata, set(metadata_table.columns))
# longitudinal_manager.run()